# WDR91 — Prospective Prediction on New Compound Library

**Goal:** Load the trained XGBoost model and use it to rank a new compound library, identifying the most likely WDR91 hits for experimental validation.

This notebook is for **Step 1** of the DREAM Target 2035 Challenge — predicting hits from the 339k retrospective validation library.

**Features used (must match training):** ECFP6 + MACCS + RDK concatenated → 4263-bit vector per compound.

---
**Before running:**
- `validation_library.parquet` → `My Drive/CS502/data/validation_library.parquet`
- Run notebook `02_train_evaluate.ipynb` first to generate the saved model
- The validation library uses `RandomID` as its compound identifier column

In [ ]:
!pip install xgboost pyarrow tqdm seaborn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

VALIDATION_PATH = '/content/drive/MyDrive/CS502/data/validation_library.parquet'
MODEL_PATH      = '/content/drive/MyDrive/CS502/models/xgb_multifp.json'
OUTPUT_PATH     = '/content/drive/MyDrive/CS502/outputs/predictions_ranked.csv'

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import xgboost as xgb
from tqdm import tqdm
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

Path('/content/drive/MyDrive/CS502/outputs').mkdir(parents=True, exist_ok=True)

## Helper: Multi-Fingerprint Loader

Loads ECFP6 + MACCS + RDK from the validation library and concatenates them into a single sparse matrix.
The fingerprints in the validation library are stored as comma-separated strings (e.g. `"123,456,789"`) — this is handled automatically.

In [ ]:
FP_DIMS = {
    'ECFP4': 2048, 'ECFP6': 2048, 'FCFP4': 2048, 'FCFP6': 2048,
    'MACCS': 167, 'RDK': 2048, 'AVALON': 512, 'ATOMPAIR': 2048, 'TOPTOR': 2048,
}

# Must match the fingerprints used during training
FP_NAMES = ['ECFP6', 'MACCS', 'RDK']
ID_COL   = 'RandomID'  # validation library uses RandomID, not COMPOUND_ID


def indices_to_sparse(index_lists, n_bits):
    """
    Convert fingerprint bit indices to a sparse CSR matrix.
    Handles both list-of-ints and comma-separated string formats.
    """
    rows, cols = [], []
    for i, bits in enumerate(index_lists):
        # Validation library stores bits as a comma-separated string
        if isinstance(bits, str):
            bits = [x.strip() for x in bits.split(',') if x.strip()]
        for b in bits:
            if int(b) < n_bits:
                rows.append(i)
                cols.append(int(b))
    data = np.ones(len(rows), dtype=np.uint8)
    return sp.csr_matrix((data, (rows, cols)), shape=(len(index_lists), n_bits))


def load_multi_fingerprints_val(path, fp_names=FP_NAMES, id_col=ID_COL, batch_size=50_000):
    """
    Load and concatenate multiple fingerprints from the validation library.
    Returns (X_combined_sparse, compound_ids).
    """
    pf = pq.ParquetFile(path)
    schema_cols = [f.name for f in pf.schema_arrow]

    cols_to_load = [c for c in [id_col] + fp_names if c in schema_cols]
    missing = [fp for fp in fp_names if fp not in schema_cols]
    if missing:
        raise ValueError(f'Missing fingerprint columns in validation library: {missing}')
    print(f'Loading columns: {cols_to_load}')

    id_lists = []
    fp_batches = {fp: [] for fp in fp_names}

    for batch in tqdm(pf.iter_batches(batch_size=batch_size, columns=cols_to_load),
                      desc='Loading fingerprints'):
        batch_df = batch.to_pandas()
        if id_col in batch_df.columns:
            id_lists.append(batch_df[id_col])
        for fp in fp_names:
            fp_batches[fp].append(indices_to_sparse(batch_df[fp].tolist(), FP_DIMS[fp]))

    fp_matrices = [sp.vstack(fp_batches[fp], format='csr') for fp in fp_names]
    X = sp.hstack(fp_matrices, format='csr')
    ids = pd.concat(id_lists, ignore_index=True) if id_lists else pd.RangeIndex(X.shape[0])

    total_bits = sum(FP_DIMS[fp] for fp in fp_names)
    print(f'Combined feature matrix: {X.shape}  ({total_bits} bits from {fp_names})')
    return X, ids

## 1. Load Model

In [ ]:
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)
print('Model loaded:', MODEL_PATH)
print(f'Expected feature vector size: {model.n_features_in_} bits')

## 2. Load Validation Library

Inspect schema first to confirm column names.

In [ ]:
pf_val = pq.ParquetFile(VALIDATION_PATH)
print('Validation schema:\n', pf_val.schema_arrow)
print(f'\nTotal compounds: {pf_val.metadata.num_rows:,}')

In [ ]:
print(f'Loading fingerprints: {FP_NAMES}...')
X_val, compound_ids = load_multi_fingerprints_val(VALIDATION_PATH)
print(f'\nX_val shape: {X_val.shape}')
print(f'Model expects: {model.n_features_in_} features')
assert X_val.shape[1] == model.n_features_in_, \
    f'Feature mismatch! Val={X_val.shape[1]}, model={model.n_features_in_}. Check FP_NAMES matches training.'

## 3. Predict & Rank

In [ ]:
print('Running predictions...')
X_val_f32 = X_val.astype(np.float32)
y_prob = model.predict_proba(X_val_f32)[:, 1]

results = pd.DataFrame({'RandomID': compound_ids, 'hit_probability': y_prob})
results = results.sort_values('hit_probability', ascending=False).reset_index(drop=True)
results['rank'] = results.index + 1

print(f'\nTop 20 predicted hits:')
print(results.head(20).to_string(index=False))

## 4. Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(y_prob, bins=100, color='steelblue', log=True)
axes[0].set_xlabel('Predicted hit probability')
axes[0].set_ylabel('Count (log)')
axes[0].set_title('Score Distribution (full library)')

top1pct = int(len(y_prob) * 0.01)
axes[1].hist(np.sort(y_prob)[-top1pct:], bins=50, color='coral')
axes[1].set_xlabel('Predicted hit probability')
axes[1].set_title(f'Top 1% scores ({top1pct:,} compounds)')

plt.tight_layout()
plt.show()

for pct in [0.1, 0.5, 1.0, 5.0]:
    n = int(len(y_prob) * pct / 100)
    threshold = np.sort(y_prob)[::-1][n-1]
    print(f'Top {pct:4.1f}% ({n:6,} compounds): score threshold = {threshold:.4f}')

## 5. If Ground Truth Labels Available — Evaluate

The Step 1 validation library does not include labels — evaluation happens server-side after submission.
This cell is a no-op for the current library but will work automatically if a labelled library is loaded.

In [ ]:
val_schema_cols = [f.name for f in pf_val.schema_arrow]

if 'LABEL' in val_schema_cols:
    labels = pq.read_table(VALIDATION_PATH, columns=['LABEL']).to_pandas()['LABEL'].values
    print(f'Ground truth: {labels.sum()} hits / {len(labels):,} total')

    def enrichment_factor(y_true, y_score, frac):
        n = len(y_true)
        n_top = max(1, int(n * frac))
        top_idx = np.argsort(y_score)[::-1][:n_top]
        hits_in_top = y_true[top_idx].sum()
        total_hits = y_true.sum()
        return 0.0 if total_hits == 0 else (hits_in_top / n_top) / (total_hits / n)

    for frac, label in [(0.001,'0.1%'), (0.005,'0.5%'), (0.01,'1%'), (0.05,'5%')]:
        ef = enrichment_factor(labels, y_prob, frac)
        n_top = int(len(labels)*frac)
        top_idx = np.argsort(y_prob)[::-1][:n_top]
        hits_found = labels[top_idx].sum()
        print(f'  EF@{label:4s}: {ef:.2f}  ({hits_found} / {labels.sum()} hits in top {n_top:,})')
else:
    print('No LABEL column — prospective prediction mode. Submit predictions_ranked.csv to the challenge portal.')

## 6. Save Submission File

In [ ]:
results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(results):,} ranked predictions to {OUTPUT_PATH}')
print(f'\nFile preview:')
results.head()